# 05 - Export Dashboard Data

## 1. Purpose
Produce the single, final, dashboard-ready CSV (`data/processed/monthly_revenue_processed.csv`)
that both the Tableau workbook and the executive report are built from. This
is the one authoritative output file for downstream consumption -- everything
upstream (SQL, EDA, growth analysis) feeds into this export.


## 2. Imports

In [1]:
import pandas as pd
import numpy as np


## 3. Data Loading

In [2]:
df = pd.read_csv("../data/raw/monthly_revenue_raw.csv")
df['sales_month'] = pd.to_datetime(df['sales_month'], format='%Y-%m')
df = df.sort_values('sales_month').reset_index(drop=True)
df.rename(columns={'current_month_revenue': 'revenue'}, inplace=True)


## 4. Analysis

In [3]:
df['previous_month_revenue'] = df['revenue'].shift(1)
df['mom_growth_pct'] = (
    (df['revenue'] - df['previous_month_revenue']) / df['previous_month_revenue'] * 100
).round(2)
df['running_revenue'] = df['revenue'].cumsum()
df['revenue_12mo_ago'] = df['revenue'].shift(12)
df['yoy_growth_pct'] = (
    (df['revenue'] - df['revenue_12mo_ago']) / df['revenue_12mo_ago'] * 100
).round(2)
df['rolling_3m_avg'] = df['revenue'].rolling(3).mean().round(2)
df['rolling_3m_std'] = df['revenue'].rolling(3).std().round(2)
df['revenue_acceleration'] = (df['mom_growth_pct'] - df['mom_growth_pct'].shift(1)).round(2)

is_decline = (df['mom_growth_pct'] < 0).astype(int)
streak_group = (np.arange(len(df)) + 1) - is_decline.cumsum()
df['consecutive_decline_months'] = np.where(
    is_decline == 1, is_decline.groupby(streak_group).cumsum(), 0
)

# NOTE: orders / order_growth_pct / aov columns are intentionally NOT
# included -- see docs/orders_and_aov_scope_note.md and
# docs/assumptions.md. Do not backfill these with estimates.

export_cols = [
    'sales_month', 'revenue', 'previous_month_revenue', 'mom_growth_pct',
    'running_revenue', 'revenue_12mo_ago', 'yoy_growth_pct',
    'rolling_3m_avg', 'rolling_3m_std', 'revenue_acceleration',
    'consecutive_decline_months'
]
export_df = df[export_cols].rename(columns={'revenue': 'current_month_revenue'})
export_df.head()


,sales_month,current_month_revenue,previous_month_revenue,mom_growth_pct,running_revenue,revenue_12mo_ago,yoy_growth_pct,rolling_3m_avg,rolling_3m_std,revenue_acceleration,consecutive_decline_months
0,2016-10-01,43000,NaN,NaN,43000,NaN,NaN,NaN,NaN,NaN,0
1,2016-11-01,105000,43000.0,144.19,148000,NaN,NaN,NaN,NaN,NaN,0
2,2016-12-01,253000,105000.0,140.95,401000,NaN,NaN,133666.67,107895.01,-3.24,0
3,2017-01-01,390000,253000.0,54.15,791000,NaN,NaN,249333.33,142535.38,-86.80,0
4,2017-02-01,362000,390000.0,-7.18,1153000,NaN,NaN,335000.00,72380.94,-61.33,1


In [4]:
OUT_PATH = "../data/processed/monthly_revenue_processed.csv"
export_df.to_csv(OUT_PATH, index=False)
print(f"Exported {len(export_df)} rows to {OUT_PATH}")
export_df.tail()


Exported 23 rows to ../data/processed/monthly_revenue_processed.csv


,sales_month,current_month_revenue,previous_month_revenue,mom_growth_pct,running_revenue,revenue_12mo_ago,yoy_growth_pct,rolling_3m_avg,rolling_3m_std,revenue_acceleration,consecutive_decline_months
18,2018-04-01,1042000,1015000.0,2.66,11608013,453000.0,130.02,972666.67,97643.91,-15.23,0
19,2018-05-01,1061000,1042000.0,1.82,12669013,526000.0,101.71,1039333.33,23115.65,-0.84,0
20,2018-06-01,1033000,1061000.0,-2.64,13702013,607000.0,70.18,1045333.33,14294.52,-4.46,1
21,2018-07-01,1038514,1033000.0,0.53,14740527,635000.0,63.55,1044171.33,14832.53,3.17,0
22,2018-08-01,996974,1038514.0,-4.00,15737501,680000.0,46.61,1022829.33,22560.47,-4.53,1


In [5]:
# Tableau-ready extract: same numbers plus helper columns (see dashboard/tableau/calculated_fields.md)
tab = export_df.copy()
n = len(tab)
rank_from_end = np.arange(n)[::-1] + 1
tab['mom_growth_decimal'] = (tab['mom_growth_pct'] / 100).round(4)
tab['growth_direction'] = np.where(tab['mom_growth_pct'].isna(), 'No prior month',
                          np.where(tab['mom_growth_pct'] > 0, 'Growth',
                          np.where(tab['mom_growth_pct'] < 0, 'Decline', 'Flat')))
tab['vs_rolling_trend'] = np.where(tab['rolling_3m_avg'].isna(), 'No trend yet',
                          np.where(tab['current_month_revenue'] >= tab['rolling_3m_avg'], 'Above trend', 'Below trend'))
tab['period_bucket'] = np.where(rank_from_end <= 3, 'Last 3 months',
                       np.where(rank_from_end <= 6, 'Previous 3 months', 'Earlier'))
tab['latest_flag'] = np.where(rank_from_end == 1, 'Latest', 'Other')
tab.to_csv("../data/processed/tableau_extract.csv", index=False)
print(f"Exported {len(tab)} rows to ../data/processed/tableau_extract.csv")
tab.tail(3)


Exported 23 rows to ../data/processed/tableau_extract.csv


,sales_month,current_month_revenue,previous_month_revenue,mom_growth_pct,running_revenue,revenue_12mo_ago,yoy_growth_pct,rolling_3m_avg,rolling_3m_std,revenue_acceleration,consecutive_decline_months,mom_growth_decimal,growth_direction,vs_rolling_trend,period_bucket,latest_flag
20,2018-06-01,1033000,1061000.0,-2.64,13702013,607000.0,70.18,1045333.33,14294.52,-4.46,1,-0.0264,Decline,Below trend,Last 3 months,Other
21,2018-07-01,1038514,1033000.0,0.53,14740527,635000.0,63.55,1044171.33,14832.53,3.17,0,0.0053,Growth,Below trend,Last 3 months,Other
22,2018-08-01,996974,1038514.0,-4.00,15737501,680000.0,46.61,1022829.33,22560.47,-4.53,1,-0.0400,Decline,Below trend,Last 3 months,Latest


## 5. Findings

- The export contains all revenue-based metrics computed and reconciled in
  `04_growth_analysis.ipynb`, at the same monthly grain as the source data
  (23 rows, one per month, Oct-2016 - Aug-2018).
- Columns intentionally **excluded**: `orders`, `order_growth_pct`, `aov`.
  These were in the original project scope but are not computable from the
  data available (see `docs/assumptions.md`).
- This file is the single source of truth for the Tableau dashboard
  (`dashboard/tableau/`) and the executive report (`reports/executive_report.md`) --
  both should be regenerated from this file if the source data changes,
  not edited independently.


## 6. Conclusion

`monthly_revenue_processed.csv` is exported and ready to connect to Tableau.
This completes the SQL -> Python -> BI pipeline for the revenue-only scope
of this engagement.
